In [0]:
%run ./common/_utils_config

In [0]:
%run ./common/_utils_reference_data

In [0]:
%run ./common/_utils_bronze_framework

In [0]:
%skip
decommission_source("public_holidays_es")

In [0]:
inventory = [describe_source_footprint(name) for name in list_source_specs()]

display(spark.createDataFrame(
    [(i["source_name"], len(i["endpoints"]), i["raw_files"], len(i["bronze_tables"]), i["log_rows"]) for i in inventory],
    "source_name STRING, endpoints INT, raw_files INT, bronze_tables INT, log_rows BIGINT",
).orderBy("source_name"))

In [0]:
%skip
for source_name in ["open_meteo", "open_meteo_forecast_malaga", "open_meteo_forecast_seville"]:
    decommission_source(source_name)

In [0]:
print("Remaining sources:", list_source_specs())

display(
    spark.table(ingestion_log_table)
    .filter(F.col("status") == "decommissioned")
    .select("source_name", "endpoint_name", "status", "file_path", "error_message", "ingested_at")
    .orderBy(F.desc("ingested_at"))
)

In [0]:
# Execution plan for today: what each source would do (read-only)
for name in list_source_specs():
    spec = load_source_spec_file(name)
    errors = validate_source_spec(spec)
    if errors:
        print(f"❌ {name}: {errors}")
        continue
    window, mode_used = resolve_window(spec, date.today())
    chunks = build_chunks(spec, spec["endpoints"][0], date.today(), window)
    unit = spec["load"].get("unit", "")
    print(f"✅ {name:20} {mode_used:8} window: {str(window or '-'):3} {unit:5} "
          f"· {len(chunks)} chunk(s): {chunks[0][0]} → {chunks[-1][1]}")

In [0]:
summary = run_source_spec(load_source_spec_file("public_holidays_es"))
print(f"\n{summary['mode']} · {summary['chunks_ok']} ok / {summary['chunks_error']} errors · tables: {list(summary['bronze_tables'])}")

display(
    spark.table(bronze_table_name("public_holidays_es", "holidays"))
    .select("chunk_start", "chunk_end",
            F.expr("array_size(cast(payload as array<variant>))").alias("holidays"),
            F.expr("payload:[0].date::string").alias("first_date"),
            F.expr("payload:[0].localName::string").alias("first_holiday"))
    .orderBy("chunk_start")
)